# Task3 v4：局部精修与整图上下文的配对对照

2026-10-27修复版：内层预测使用与独立粗中心一致的fold=-1，外层仍使用真实折号；修复首轮融合报错，并消除预检查loss转标量警告。默认输出改为独立的task3_context_refine_v4_fixed，避免混入旧版代码指纹。

官方指标已确认：X/Y坐标行的RMSE，越小越好。v3预设候选OOF16.8015，固定粗基线16.5282；公开最好仍是全量320 ResNet34的12.31624。本版尚未训练，不保证提升。

本轮只比较两个配对组：**local**为局部384热图网络；**context**额外读取256整图，共享同一ImageNet ResNet34骨干，把整图对应区域特征及全局池化特征加到局部网络。相同局部数据、损失、位置先验、增强、批次种子及内层门槛；增加的融合投影从零初始化，使起点等价。

参考[ZIAN关于上下文特征融合的研究](https://arxiv.org/html/2210.02445v2)。这是简化的共享骨干加法融合对照，不是ZIAN完整复现，未使用外部眼底标签或其训练权重。


## 1. 运行配置

导入本Notebook，添加2026task3比赛数据，开启GPU和Internet，运行全部单元。已内置43个CSV/JSON文件：上轮512粗预测，以及v3内层独立辅助粗模型的预测和来源记录。无需再上传ZIP或权重，也不重训5个辅助粗模型。

完整最多20个精修训练阶段：每折两个内层60轮拟合，再按内层选择轮次进行最多两个外层重训。只用cuda:0，阶段顺序运行。按旧T4日志及新增整图计算量粗估40～75分钟，实际受门控、选中轮次及备份压缩影响，尚无本版实测时间。


In [ ]:
from pathlib import Path
if not Path('/kaggle/working').is_dir():
    raise RuntimeError('仅在Kaggle GPU运行；本机不导入Torch或训练。')
DATA_DIR=None
SMOKE_TEST=False
OUTPUT_DIR=Path('/kaggle/working/task3_context_refine_v4_fixed'+('_smoke' if SMOKE_TEST else ''))
RESUME_DIR=None  # 本版完整Output目录，直接含本版protocol.json
WEIGHTS=None
RUN_FOLDS=[0] if SMOKE_TEST else [0,1,2,3,4]
CREATE_FULL_BACKUP=True


In [ ]:
from __future__ import annotations
from collections import defaultdict
from contextlib import contextmanager
import base64, csv, gc, hashlib, io, json, math, os, random, shutil, statistics, sys, time, zipfile
import xml.etree.ElementTree as ET
from pathlib import Path, PurePosixPath
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
WEIGHT_NAME='resnet34-b627a593.pth'
WEIGHT_URL='https://download.pytorch.org/models/'+WEIGHT_NAME

def save_json(path, value):
    path = Path(path)
    tmp = path.with_suffix('.tmp')
    tmp.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding='utf-8')
    tmp.replace(path)


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.set_num_threads(4)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def find_data(explicit=None):
    if explicit:
        candidates = [Path(explicit)]
    else:
        base = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
        candidates = [base / '2026task3/detection', Path.cwd() / '2026task3/detection',
                      Path.cwd() / 'task3/2026task3/detection']
        if Path('/kaggle/input').exists():
            candidates += [p.parent for p in Path('/kaggle/input').rglob('fovea_localization_train_GT.csv')]
    valid = sorted({p.resolve() for p in candidates
                    if (p / 'fovea_localization_train_GT.csv').is_file()
                    and (p / 'sample_submission.csv').is_file()
                    and (p / 'train').is_dir() and (p / 'test').is_dir()})
    if len(valid) != 1:
        raise ValueError(f'Expected one dataset; set data_dir explicitly. Found: {valid}')
    return valid[0]


def load_manifest(data):
    labels = pd.read_csv(data / 'fovea_localization_train_GT.csv')
    if list(labels.columns) != ['data', 'Fovea_X', 'Fovea_Y'] or labels.isna().any().any():
        raise ValueError('Invalid training CSV columns or missing values.')
    if labels['data'].duplicated().any():
        raise ValueError('Duplicate training IDs.')
    lookup = labels.set_index('data')
    records = []
    for subset in ['train', 'test']:
        files = sorted(p for p in (data / subset).iterdir()
                       if p.suffix.lower() in {'.jpg', '.jpeg', '.png'})
        for p in files:
            image_id = int(p.stem)
            with Image.open(p) as im:
                im = im.convert('RGB')
                im.load()
                w, h = im.size
                # Hash decoded pixels, not JPEG metadata. Size participates in the hash.
                digest = hashlib.sha256(f'{w},{h}:'.encode() + im.tobytes()).hexdigest()
            record = dict(image_id=image_id, subset=subset, filename=p.name,
                          width=w, height=h, pixel_sha256=digest)
            if subset == 'train':
                if image_id not in lookup.index:
                    raise ValueError(f'Missing label: {p.name}')
                x, y = lookup.loc[image_id, ['Fovea_X', 'Fovea_Y']].astype(float)
                if not np.isfinite([x, y]).all() or not (0 <= x < w and 0 <= y < h):
                    raise ValueError(f'Coordinate outside image: {p.name}')
                record.update(x=float(x), y=float(y), nx=float(x / w), ny=float(y / h))
                xml_path = data / 'train_location' / f'{image_id:04}.xml'
                if not xml_path.is_file():
                    raise ValueError(f'Missing XML: {xml_path}')
                xml = ET.parse(xml_path).getroot()
                if (int(xml.findtext('size/width')), int(xml.findtext('size/height'))) != (w, h):
                    raise ValueError(f'XML size mismatch: {p.name}')
                objects = [v for v in xml.findall('object') if v.findtext('name').lower() == 'fovea']
                if len(objects) != 1:
                    raise ValueError(f'Expected one Fovea box: {xml_path}')
                box = objects[0].find('bndbox')
                xmin, ymin, xmax, ymax = [float(box.findtext(k)) for k in ['xmin', 'ymin', 'xmax', 'ymax']]
                if not (xmin <= x <= xmax and ymin <= y <= ymax):
                    raise ValueError(f'CSV coordinate outside XML box: {p.name}')
                record.update(xml_center_dx=float(x - (xmin + xmax) / 2),
                              xml_center_dy=float(y - (ymin + ymax) / 2))
            records.append(record)
    manifest = pd.DataFrame(records)
    if set(manifest.loc[manifest.subset == 'train', 'image_id']) != set(labels['data']):
        raise ValueError('Training images and CSV IDs differ.')
    if manifest.image_id.duplicated().any():
        raise ValueError('Image IDs overlap between train/test.')
    cross = manifest.groupby('pixel_sha256').subset.nunique()
    if (cross > 1).any():
        raise ValueError('Identical pixels occur in both train and test; inspect before continuing.')
    return manifest


def assign_folds(train, seed=42, folds=5):
    """Balance exact-duplicate groups across image resolutions; no test labels used."""
    if folds < 2:
        raise ValueError('Need at least two folds.')
    train = train.copy()
    groups = list(train.groupby('pixel_sha256', sort=True))
    rng = np.random.default_rng(seed)
    buckets = defaultdict(list)
    for digest, group in groups:
        row = group.iloc[0]
        buckets[(int(row.width), int(row.height))].append((digest, len(group)))
    total = np.zeros(folds, dtype=int)
    mapping = {}
    for key in sorted(buckets):
        entries = buckets[key]
        rng.shuffle(entries)
        entries.sort(key=lambda pair: -pair[1])
        sizes = np.zeros(folds, dtype=int)
        for digest, count in entries:
            tie = rng.random(folds)
            fold = min(range(folds), key=lambda f: (sizes[f], total[f], tie[f]))
            mapping[digest] = fold
            sizes[fold] += count
            total[fold] += count
    train['fold'] = train.pixel_sha256.map(mapping)
    if train.groupby('pixel_sha256').fold.nunique().max() != 1:
        raise AssertionError('Duplicate group split across folds.')
    return train


class DecoderBlock(nn.Module):
    def __init__(self, incoming, skip, outgoing):
        super().__init__()
        self.layers = nn.Sequential(nn.Conv2d(incoming + skip, outgoing, 3, padding=1, bias=False),
                                    nn.GroupNorm(8, outgoing), nn.ReLU(inplace=True),
                                    nn.Conv2d(outgoing, outgoing, 3, padding=1, bias=False),
                                    nn.GroupNorm(8, outgoing), nn.ReLU(inplace=True))

    def forward(self, value, skip):
        value = F.interpolate(value, size=skip.shape[-2:], mode='bilinear', align_corners=False)
        return self.layers(torch.cat([value, skip], dim=1))


def heatmap_loss(logits, points, sigma=1.5):
    height, width = logits.shape[-2:]
    yy = torch.arange(height, device=logits.device, dtype=torch.float32) + .5
    xx = torch.arange(width, device=logits.device, dtype=torch.float32) + .5
    distance = (xx[None, None, :] - points[:, 0, None, None] * width) ** 2
    distance = distance + (yy[None, :, None] - points[:, 1, None, None] * height) ** 2
    target = torch.exp(-distance / (2 * sigma ** 2))
    target = target / target.sum(dim=(-2, -1), keepdim=True).clamp_min(1e-12)
    return -(target.flatten(1) * F.log_softmax(logits.float().flatten(1), dim=1)).sum(1).mean()


def decode_heatmap(logits, radius=3):
    """Local expectation around the peak; cell centers match resize align_corners=False."""
    batch, height, width = logits.shape
    peak = logits.flatten(1).argmax(1)
    py, px = peak // width, peak % width
    yy = torch.arange(height, device=logits.device)[None, :, None]
    xx = torch.arange(width, device=logits.device)[None, None, :]
    mask = (yy - py[:, None, None]).abs() <= radius
    mask = mask & ((xx - px[:, None, None]).abs() <= radius)
    weights = logits.float().masked_fill(~mask, -torch.inf).flatten(1).softmax(1).view(batch, height, width)
    x = (weights * ((xx + .5) / width)).sum((-2, -1))
    y = (weights * ((yy + .5) / height)).sum((-2, -1))
    return torch.stack([x, y], dim=1)


@contextmanager
def training_lock(out):
    handle = (out / 'training.lock').open('a+b')
    if handle.tell() == 0:
        handle.write(b'0')
        handle.flush()
    handle.seek(0)
    try:
        if os.name == 'nt':
            import msvcrt
            msvcrt.locking(handle.fileno(), msvcrt.LK_NBLCK, 1)
        else:
            import fcntl
            fcntl.flock(handle.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)
        yield
    finally:
        handle.close()


def stable_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False,
                                     separators=(',', ':'), allow_nan=False).encode('utf-8')).hexdigest()


def atomic_csv(path, rows, fields=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    rows = list(rows)
    fields = list(fields or (list(rows[0]) if rows else []))
    temporary = path.with_suffix(path.suffix + '.tmp')
    with temporary.open('w', encoding='utf-8', newline='') as file:
        writer = csv.DictWriter(file, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)
    temporary.replace(path)


def read_rows(path):
    with Path(path).open(encoding='utf-8', newline='') as file:
        return list(csv.DictReader(file))


def quantile_linear(values, probability):
    values = sorted(float(v) for v in values)
    if not values:
        raise ValueError('No values.')
    position = (len(values) - 1) * probability
    low, high = math.floor(position), math.ceil(position)
    return values[low] + (values[high] - values[low]) * (position - low)


def metrics_from_rows(rows):
    rows = list(rows)
    if not rows:
        raise ValueError('No labeled predictions.')
    distance, dx, dy = [], [], []
    for row in rows:
        ex = float(row['Fovea_X']) - float(row['gt_x'])
        ey = float(row['Fovea_Y']) - float(row['gt_y'])
        if not math.isfinite(ex) or not math.isfinite(ey):
            raise ValueError('Non-finite error.')
        dx.append(ex)
        dy.append(ey)
        distance.append(math.hypot(ex, ey))
    return dict(n=len(rows), mean_distance_px=statistics.mean(distance),
                median_distance_px=statistics.median(distance),
                p90_distance_px=quantile_linear(distance, .9), max_distance_px=max(distance),
                mae_x_px=statistics.mean(abs(v) for v in dx), mae_y_px=statistics.mean(abs(v) for v in dy),
                rmse_coordinate_px=math.sqrt(sum(x*x+y*y for x, y in zip(dx, dy)) / (2 * len(rows))))


def validate_oof(rows, records):
    expected = {int(r['image_id']): r for r in records}
    if len(expected) != len(records) or len(rows) != len(records):
        raise ValueError('OOF must contain exactly one row per training image.')
    seen, groups = set(), {}
    for row in rows:
        key = int(row['image_id'])
        if key not in expected or key in seen:
            raise ValueError('Unknown or repeated OOF image ID.')
        seen.add(key)
        source = expected[key]
        if int(row['fold']) != int(source['fold']):
            raise ValueError('OOF fold does not match the fixed split.')
        if (int(row['width']), int(row['height'])) != (int(source['width']), int(source['height'])):
            raise ValueError('OOF dimensions differ.')
        if str(row['pixel_sha256']) != str(source['pixel_sha256']):
            raise ValueError('OOF image hash differs.')
        if abs(float(row['gt_x']) - float(source['x'])) > 1e-6 or abs(float(row['gt_y']) - float(source['y'])) > 1e-6:
            raise ValueError('OOF ground truth differs.')
        groups.setdefault(str(source['pixel_sha256']), set()).add(int(row['fold']))
    if seen != set(expected) or any(len(value) != 1 for value in groups.values()):
        raise ValueError('Incomplete OOF coverage or duplicate group leakage.')
    return sorted(rows, key=lambda r: int(r['image_id']))


def validate_test_rows(rows, records, fold):
    expected = {int(r['image_id']): r for r in records}
    if len(rows) != len(expected):
        raise ValueError('Incomplete test prediction coverage.')
    seen = set()
    for row in rows:
        key = int(row['image_id'])
        if key not in expected or key in seen or int(row['fold']) != fold:
            raise ValueError('Invalid test ID/fold.')
        seen.add(key)
        source = expected[key]
        if (int(row['width']), int(row['height'])) != (int(source['width']), int(source['height'])):
            raise ValueError('Test image dimensions differ.')
        if str(row['pixel_sha256']) != str(source['pixel_sha256']):
            raise ValueError('Test image hash differs.')
        x, y = float(row['Fovea_X']), float(row['Fovea_Y'])
        if not all(math.isfinite(v) for v in [x, y]) or not (0 <= x < int(source['width']) and 0 <= y < int(source['height'])):
            raise ValueError('Test prediction outside image.')
    return {int(r['image_id']): r for r in rows}


def average_test_folds(folds, records, fold_count=5):
    if set(folds) != set(range(fold_count)):
        raise ValueError('All five folds are required before exporting a five-fold submission.')
    mappings = [validate_test_rows(folds[f], records, f) for f in range(fold_count)]
    averaged = []
    for source in records:
        key = int(source['image_id'])
        averaged.append(dict(image_id=key, width=int(source['width']), height=int(source['height']),
                             Fovea_X=statistics.mean(float(m[key]['Fovea_X']) for m in mappings),
                             Fovea_Y=statistics.mean(float(m[key]['Fovea_Y']) for m in mappings)))
    return averaged


def write_submission(template_path, predictions, target):
    lookup = {}
    for row in predictions:
        key = int(row['image_id'])
        for axis, dimension in [('X', 'width'), ('Y', 'height')]:
            value = float(row[f'Fovea_{axis}'])
            if not math.isfinite(value) or not 0 <= value < int(row[dimension]):
                raise ValueError('Invalid original-pixel coordinate.')
            image_id = f'{key}_Fovea_{axis}'
            if image_id in lookup:
                raise ValueError('Repeated submission ID.')
            lookup[image_id] = value
    template = read_rows(template_path)
    ids = [row['ImageID'] for row in template]
    if len(ids) != len(set(ids)) or set(ids) != set(lookup):
        raise ValueError('Submission IDs differ from template.')
    rows = [dict(ImageID=key, value=f'{lookup[key]:.6f}') for key in ids]
    atomic_csv(target, rows, ['ImageID', 'value'])
    check = read_rows(target)
    if [r['ImageID'] for r in check] != ids or any(not math.isfinite(float(r['value'])) for r in check):
        raise AssertionError('Submission round trip failed.')


def resolve_weights(output, explicit=None):
    candidates = [Path(explicit)] if explicit else []
    candidates += [Path(torch.hub.get_dir()) / 'checkpoints' / WEIGHT_NAME]
    if '__file__' in globals():
        candidates.append(Path(__file__).resolve().parent.parent / 'task2/.torch/hub/checkpoints' / WEIGHT_NAME)
    if Path('/kaggle/input').exists():
        candidates += sorted(Path('/kaggle/input').rglob(WEIGHT_NAME))
    path = next((p for p in candidates if p.is_file()), None)
    if path is None:
        path = output / '_pretrained' / WEIGHT_NAME
        path.parent.mkdir(parents=True, exist_ok=True)
        if not path.is_file():
            print('Downloading official ResNet34 ImageNet weights (about 87 MB).', flush=True)
            torch.hub.download_url_to_file(WEIGHT_URL, str(path), hash_prefix='b627a593', progress=True)
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    if not digest.startswith('b627a593'):
        raise ValueError(f'ImageNet weight hash mismatch: {path}')
    save_json(output / 'pretrained_source.json', dict(path=str(path), url=WEIGHT_URL, sha256=digest))
    return path


class ResNet34CV(nn.Module):
    def __init__(self, weights=None):
        super().__init__()
        self.backbone = models.resnet34(weights=None)
        if weights is not None:
            self.backbone.load_state_dict(torch.load(weights, map_location='cpu', weights_only=True), strict=True)
        self.backbone.fc = nn.Identity()
        self.d3 = DecoderBlock(512, 256, 128)
        self.d2 = DecoderBlock(128, 128, 64)
        self.d1 = DecoderBlock(64, 64, 32)
        self.head = nn.Conv2d(32, 1, 1)

    def train(self, mode=True):
        super().train(mode)
        if mode:
            for module in self.backbone.modules():
                if isinstance(module, nn.BatchNorm2d):
                    module.eval()
        return self

    def forward(self, images):
        b = self.backbone
        x = b.maxpool(b.relu(b.bn1(b.conv1(images))))
        c1 = b.layer1(x)
        c2 = b.layer2(c1)
        c3 = b.layer3(c2)
        c4 = b.layer4(c3)
        return self.head(self.d1(self.d2(self.d3(c4, c3), c2), c1))[:, 0]


def atomic_torch_save(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.tmp')
    torch.save(value, temporary)
    temporary.replace(path)


def restore_backup(source, output, protocol):
    if source is None:
        return
    source = Path(source).resolve()
    if source == output.resolve():
        return
    candidate = source / 'protocol.json'
    if not candidate.is_file() or json.loads(candidate.read_text()) != protocol:
        raise ValueError('Backup protocol/data/code differs. Choose the correct prior output folder.')
    if (output / 'protocol.json').exists():
        print('Current output already has progress; resuming it without merging backup files.', flush=True)
        return
    for path in source.rglob('*'):
        if not path.is_file() or path.is_symlink() or path.suffix in {'.tmp', '.zip'}:
            continue
        if not path.resolve().is_relative_to(source):
            raise ValueError('Backup path escapes its source directory.')
        destination = output / path.relative_to(source)
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(path, destination)


class CoarseBundle:
    """Read only metadata/predictions from a directory or ZIP; never extract/load weights."""
    def __init__(self, path):
        self.path, self.prefix, self.archive = Path(path).resolve(), '', None
        if self.path.is_dir():
            return
        if not self.path.is_file() or self.path.suffix.lower() != '.zip':
            raise ValueError(f'Coarse source is not a directory or ZIP: {path}')
        self.archive = zipfile.ZipFile(self.path)
        try:
            candidates = []
            for entry in self.archive.infolist():
                pure = PurePosixPath(entry.filename)
                if pure.is_absolute() or '..' in pure.parts or '\\' in entry.filename:
                    raise ValueError('Unsafe archive member path.')
                if pure.name == 'protocol.json' and entry.file_size < 1000000:
                    value = json.loads(self.archive.read(entry).decode('utf-8-sig'))
                    if value.get('version') == 'resnet34-cv-v1':
                        candidates.append(entry.filename[:-len('protocol.json')])
            if len(candidates) != 1:
                raise ValueError('ZIP must contain exactly one original CV protocol.')
            self.prefix = candidates[0]
        except BaseException:
            self.close()
            raise

    def text(self, relative):
        if self.archive is None:
            path = (self.path / relative).resolve()
            if not path.is_relative_to(self.path):
                raise ValueError('Coarse file escapes its source folder.')
            if path.stat().st_size > 5000000:
                raise ValueError('Unexpectedly large metadata file.')
            return path.read_text(encoding='utf-8-sig')
        name = self.prefix + relative
        if self.archive.getinfo(name).file_size > 5000000:
            raise ValueError('Unexpectedly large metadata member.')
        return self.archive.read(name).decode('utf-8-sig')

    def rows(self, name):
        return list(csv.DictReader(io.StringIO(self.text(name), newline='')))

    def json(self, name):
        return json.loads(self.text(name))

    def close(self):
        if self.archive is not None:
            self.archive.close()
            self.archive = None


def validate_coarse_bundle(source, train_records, test_records):
    bundle = CoarseBundle(source)
    try:
        protocol = bundle.json('protocol.json')
        if protocol.get('version') != 'resnet34-cv-v1' or protocol.get('folds') != 5:
            raise ValueError('Expected the previous five-fold CV experiment.')
        if protocol.get('selection_metric') != 'rmse_coordinate_px':
            raise ValueError('This predeclared experiment reuses the RMSE-selected 512 coarse checkpoints.')
        old_split = bundle.rows('split.csv')
        if len(old_split) != len(train_records):
            raise ValueError('Previous split is incomplete.')
        prior = {int(r['image_id']): r for r in old_split}
        if len(prior) != len(train_records):
            raise ValueError('Duplicate IDs in previous split.')
        for record in train_records:
            old = prior[int(record['image_id'])]
            for field in ('fold', 'width', 'height'):
                if int(old[field]) != int(record[field]):
                    raise ValueError('Coarse split or dimensions differ.')
            if old['pixel_sha256'] != record['pixel_sha256']:
                raise ValueError('Coarse image hash differs.')
            if any(abs(float(old[k])-float(record[k])) > 1e-6 for k in ('x', 'y')):
                raise ValueError('Coarse labels differ.')
        validation, testing, metadata, coverage = {}, {}, [], []
        for fold in range(5):
            prefix = f'resnet34_512/fold_{fold}/'
            config = bundle.json(prefix+'train_config.json')
            done = bundle.json(prefix+'fold_complete.json')
            signature = stable_hash(config)
            if (int(config['fold']) != fold or int(config['image_size']) != 512 or
                    config['data_fingerprint'] != protocol['data_fingerprint'] or
                    config['source_sha256'] != protocol['source_sha256'] or
                    done.get('signature') != signature or not done.get('predictions_complete') or
                    done['training']['signature'] != signature):
                raise ValueError('Coarse fold metadata is inconsistent.')
            held = [r for r in train_records if int(r['fold']) == fold]
            rows = validate_oof(bundle.rows(prefix+'validation_raw.csv'), held)
            # Verify finite in-bounds coordinates as well as OOF membership/labels.
            validate_test_rows(rows, held, fold)
            best = done['training']['best']['rmse_coordinate_px']
            if not math.isclose(metrics_from_rows(rows)['rmse_coordinate_px'], float(best['score']), abs_tol=1e-6):
                raise ValueError('Coarse predictions do not match the selected checkpoint metrics.')
            validation[fold] = {int(r['image_id']): r for r in rows}
            testing[fold] = validate_test_rows(bundle.rows(prefix+'test_raw.csv'), test_records, fold)
            metadata.append(dict(fold=fold, config=config, selected=best))
            coverage.extend(rows)
        fingerprint = stable_hash(dict(protocol=protocol, split=old_split, folds=metadata,
                                        validation=validation, test=testing))
        return dict(validation=validation, test=testing, fingerprint=fingerprint,
                    protocol=protocol, metadata=metadata,
                    baseline_metrics=metrics_from_rows(coverage))
    finally:
        bundle.close()


def crop_geometry(center, side):
    if side < 1 or int(side) != side or not all(math.isfinite(float(v)) for v in center):
        raise ValueError('Invalid crop geometry.')
    left, top = [math.floor(float(v)-side/2) for v in center]
    return (left, top, left+int(side), top+int(side))


def point_to_patch(point, box):
    return ((float(point[0])-box[0])/(box[2]-box[0]),
            (float(point[1])-box[1])/(box[3]-box[1]))


def patch_to_original(point, box):
    return (box[0]+float(point[0])*(box[2]-box[0]),
            box[1]+float(point[1])*(box[3]-box[1]))


def make_refinement_row(record, point, coarse, box, fold, max_shift):
    """Map patch coordinates back before scoring; a fixed fallback needs no labels."""
    width, height = int(record['width']), int(record['height'])
    cx, cy = float(coarse['Fovea_X']), float(coarse['Fovea_Y'])
    if not (0 <= cx < width and 0 <= cy < height):
        raise ValueError('Coarse coordinate outside original image.')
    raw_x, raw_y = patch_to_original(point, box)
    shift = math.hypot(raw_x-cx, raw_y-cy)
    reason = ''
    if not all(math.isfinite(v) for v in (raw_x, raw_y)):
        raise ValueError('Non-finite refinement coordinate.')
    if not (0 <= raw_x < width and 0 <= raw_y < height):
        reason = 'outside_image'
    elif shift > max_shift:
        reason = 'shift_gt_fixed_limit'
    x, y = (cx, cy) if reason else (raw_x, raw_y)
    row = dict(image_id=int(record['image_id']), fold=int(fold), width=width, height=height,
               pixel_sha256=str(record['pixel_sha256']), nx=x/width, ny=y/height,
               Fovea_X=x, Fovea_Y=y, coarse_x=cx, coarse_y=cy,
               raw_refined_x=raw_x, raw_refined_y=raw_y, shift_px=shift,
               fallback=int(bool(reason)), fallback_reason=reason,
               crop_left=box[0], crop_top=box[1], crop_side=box[2]-box[0])
    if record['subset'] == 'train':
        dx, dy = x-float(record['x']), y-float(record['y'])
        row.update(gt_x=float(record['x']), gt_y=float(record['y']), dx=dx, dy=dy,
                   distance_px=math.hypot(dx, dy), squared_error=dx*dx+dy*dy)
    return row


def blend_prediction_rows(coarse_rows, refine_rows, labeled):
    coarse = {int(r['image_id']): r for r in coarse_rows}
    refined = {int(r['image_id']): r for r in refine_rows}
    if len(coarse) != len(coarse_rows) or len(refined) != len(refine_rows) or set(coarse) != set(refined):
        raise ValueError('Blend IDs differ or repeat.')
    result = []
    for key in sorted(coarse):
        a, b = coarse[key], refined[key]
        for field in ('fold', 'width', 'height'):
            if int(a[field]) != int(b[field]):
                raise ValueError('Blend fold or dimensions differ.')
        if a['pixel_sha256'] != b['pixel_sha256']:
            raise ValueError('Blend pixels differ.')
        row = {name:a[name] for name in ('image_id', 'fold', 'width', 'height', 'pixel_sha256')}
        x, y = [(float(a[f'Fovea_{axis}'])+float(b[f'Fovea_{axis}']))/2 for axis in ('X','Y')]
        row.update(Fovea_X=x, Fovea_Y=y, nx=x/int(a['width']), ny=y/int(a['height']))
        if labeled:
            if any(abs(float(a[k])-float(b[k])) > 1e-6 for k in ('gt_x','gt_y')):
                raise ValueError('Blend ground truth differs.')
            dx, dy = x-float(a['gt_x']), y-float(a['gt_y'])
            row.update(gt_x=float(a['gt_x']), gt_y=float(a['gt_y']), dx=dx, dy=dy,
                       distance_px=math.hypot(dx,dy), squared_error=dx*dx+dy*dy)
        result.append(row)
    return result


class OriginalImageCache:
    def __init__(self, data):
        self.data, self.images = Path(data), {}

    def get(self, row):
        key = int(row['image_id'])
        if key not in self.images:
            with Image.open(self.data/row['subset']/row['filename']) as image:
                self.images[key] = image.convert('RGB')
        image = self.images[key]
        if image.size != (int(row['width']), int(row['height'])):
            raise ValueError('Original image dimensions changed.')
        return image


def inner_partition(train,outer_fold):
    development=train[train.fold!=outer_fold].copy()
    outer=train[train.fold==outer_fold].copy()
    assigned=assign_folds(development,7000+outer_fold,5)
    validation_ids=set(int(x) for x in assigned.loc[assigned.fold==0,'image_id'])
    fit=development[~development.image_id.isin(validation_ids)].copy()
    validation=development[development.image_id.isin(validation_ids)].copy()
    sets=[set(frame.pixel_sha256) for frame in (fit,validation,outer)]
    if not len(fit) or not len(validation) or any(sets[i]&sets[j] for i,j in ((0,1),(0,2),(1,2))):
        raise ValueError('Inner/outer image groups overlap or are empty.')
    if set(fit.image_id)|set(validation.image_id)!=set(development.image_id):
        raise ValueError('Inner split does not cover the outer training images.')
    return development,fit,validation,outer


def center_offset(arm,uniforms):
    u,v,tail=[float(x) for x in uniforms]
    if not all(0<x<1 for x in (u,v,tail)):raise ValueError('Uniforms must be strictly between0 and1.')
    if arm=='wide':return ((2*u-1)*160,(2*v-1)*160)
    if arm!='small':raise ValueError('Unknown training arm.')
    if tail<.1:return ((2*u-1)*64,(2*v-1)*64)
    normal=statistics.NormalDist()
    return (max(-48.,min(48.,normal.inv_cdf(u)*16)),max(-48.,min(48.,normal.inv_cdf(v)*16)))


def selected_inner_epoch(best,baseline,min_gain=.05):
    score=float(best['score']);base=float(baseline)
    if not math.isfinite(score+base) or base<0 or int(best['epoch'])<1:
        raise ValueError('Invalid inner selection metrics.')
    enabled=base>0 and score<=(1-min_gain)*base
    return dict(enabled=enabled,epoch=int(best['epoch']) if enabled else 0,
                inner_best_epoch=int(best['epoch']),inner_score=score,inner_baseline=base,
                min_relative_gain=min_gain,reason='inner_gain_passed' if enabled else 'inner_gain_below_fixed_margin')


def coarse_log_prior(logits,boxes,coarse_points,sigma_native=48.):
    if sigma_native<=0:raise ValueError('Prior width must be positive.')
    height,width=logits.shape[-2:]
    # Pixel centers use the same convention as decode_heatmap and PIL resize.
    box=boxes.to(logits.device,dtype=torch.float32)
    points=torch.as_tensor(coarse_points,device=logits.device,dtype=torch.float32)
    xs=box[:,0,None,None]+(torch.arange(width,device=logits.device)[None,None,:]+.5)*(768/width)
    ys=box[:,1,None,None]+(torch.arange(height,device=logits.device)[None,:,None]+.5)*(768/height)
    distance=(xs-points[:,0,None,None])**2+(ys-points[:,1,None,None])**2
    return logits.float()-distance/(2*sigma_native**2)


def raw_prediction_rows(rows):
    result=[]
    for source in rows:
        row=dict(source)
        x,y=float(row['raw_refined_x']),float(row['raw_refined_y'])
        row.update(Fovea_X=x,Fovea_Y=y,nx=x/int(row['width']),ny=y/int(row['height']))
        if 'gt_x' in row:
            dx,dy=x-float(row['gt_x']),y-float(row['gt_y'])
            row.update(dx=dx,dy=dy,distance_px=math.hypot(dx,dy),squared_error=dx*dx+dy*dy)
        result.append(row)
    return result


def inner_mode_metrics(refined,centers):
    coarse=[centers[int(r['image_id'])] for r in refined]
    equal=blend_prediction_rows(coarse,refined,True)
    raw=raw_prediction_rows(refined)
    return dict(score=metrics_from_rows(equal)['rmse_coordinate_px'],
        refined_rmse=metrics_from_rows(refined)['rmse_coordinate_px'],
        raw_rmse=metrics_from_rows(raw)['rmse_coordinate_px'],
        mean_distance=metrics_from_rows(equal)['mean_distance_px'],
        fallbacks=sum(int(r['fallback']) for r in refined))


def validate_inner_centers(rows,records):
    mapping=validate_test_rows(rows,records,-1)
    for r in records:
        p=mapping[int(r['image_id'])]
        if abs(float(p['gt_x'])-float(r['x']))>1e-6 or abs(float(p['gt_y'])-float(r['y']))>1e-6:
            raise ValueError('Inner center labels differ.')
    return mapping


def identity_rows(records,centers,fold,selection):
    rows=[]
    for r in records:
        c=centers[int(r['image_id'])];point=(float(c['Fovea_X']),float(c['Fovea_Y']))
        box=crop_geometry(point,768)
        row=make_refinement_row(r,point_to_patch(point,box),c,box,fold,256.)
        row.update(decoder='identity',refine_applied=0,selection_reason=selection['reason'])
        rows.append(row)
    return rows


In [ ]:
from __future__ import annotations
CONTEXT_SOURCE_SHA256={'task3_pipeline.py': '5a27db4b7901d185eec831080a16d783ad001adc22b639474d345fe7b315ec3f', 'task3_kaggle_cv.py': '2227703a35b3955b8ce02801930b0bf496703decece4ad5234a1d1f70a0b12e6', 'task3_kaggle_refine.py': 'feed93ec81a810165a968715b55155f381f987f13920777819a688e55e8822cb', 'task3_refine_diagnostic_v3.py': '72387a1d8c0636767aa3881f636c4241136ce52367d49ebd8301fabeb9b0cf26', 'task3_context_refine_v4.py': 'e57e7485614f70c29a92df283814dcf7cfe0afaa9cd62b14997bf55263ce52a8'}
CONTEXT_CACHE_SHA256='d77a135fb286608f2c8c7aace35b7ccb6d5d6710e860d33696b6aae9cc3a97b6'
"""Paired local/global-context refinement, using frozen coarse-prediction caches."""
import base64
import csv
import gc
import hashlib
import io
import json
import math
from pathlib import Path, PurePosixPath
import random
import shutil
import statistics
import time
import zipfile


import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

CONTEXT_VERSION = 'context-refine-v4'
CONTEXT_ARMS = ('local', 'context')
V3_SOURCE_SHA = '72387a1d8c0636767aa3881f636c4241136ce52367d49ebd8301fabeb9b0cf26'


def unpack_context_inputs(encoded, digest, output):
    payload = base64.b64decode(encoded, validate=True)
    if hashlib.sha256(payload).hexdigest() != digest:raise ValueError('Embedded cache checksum differs.')
    output = Path(output).resolve(); output.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(payload)) as archive:
        names = archive.namelist()
        if len(names) != len(set(names)):raise ValueError('Repeated cache member.')
        for item in archive.infolist():
            path = PurePosixPath(item.filename)
            target = (output/item.filename).resolve()
            if path.is_absolute() or '..' in path.parts or '\\' in item.filename or not target.is_relative_to(output):
                raise ValueError('Cache path escapes output.')
            if item.file_size > 5000000 or path.suffix not in ('.json', '.csv'):raise ValueError('Unexpected cache member.')
        for item in archive.infolist():
            target = output/item.filename; target.parent.mkdir(parents=True, exist_ok=True)
            data = archive.read(item)
            if target.exists() and target.read_bytes() != data:raise ValueError('Existing cache differs; use a clean cache directory.')
            if not target.exists():target.write_bytes(data)
    return output


def validate_context_centers(source, train, coarse_fingerprint):
    source = Path(source)
    old_protocol = json.loads((source/'inner/protocol.json').read_text())
    if (old_protocol['version'] != 'refine-diagnostic-v3' or old_protocol['smoke_test'] or
            old_protocol['inner_coarse_epochs'] != 43 or old_protocol['coarse_fingerprint'] != coarse_fingerprint or
            old_protocol['source_sha256']['task3_refine_diagnostic_v3.py'] != V3_SOURCE_SHA):
        raise ValueError('Expected the completed v3 auxiliary coarse predictions.')
    centers, provenance = {}, []
    for fold in range(5):
        dev, fit, held, outer = inner_partition(train, fold)
        folder = source/f'inner/outer_{fold}'
        design = json.loads((folder/'design.json').read_text())
        expected = dict(protocol=old_protocol, fold=fold, fit_ids=list(map(int,fit.image_id)),
            inner_val_ids=list(map(int,held.image_id)), outer_ids=list(map(int,outer.image_id)))
        if design != expected:raise ValueError('Cached inner split differs.')
        cfg = json.loads((folder/'config.json').read_text());done = json.loads((folder/'coarse_complete.json').read_text())
        if (cfg['train_ids'] != expected['fit_ids'] or cfg['inner_validation_ids'] != expected['inner_val_ids'] or
                cfg['forbidden_outer_ids'] != expected['outer_ids'] or cfg['epochs'] != 43 or cfg['schedule_epochs'] != 80 or
                cfg['source_sha256'] != old_protocol['source_sha256'] or cfg['coarse_fingerprint'] != coarse_fingerprint or
                done['signature'] != stable_hash(cfg) or done['epoch'] != 43 or done['validation_used_for_training']):
            raise ValueError('Cached auxiliary model provenance differs.')
        predictions = read_rows(folder/'inner_validation.csv')
        centers[fold] = validate_inner_centers(predictions,held.to_dict('records'))
        actual = metrics_from_rows(predictions)
        if not math.isclose(actual['rmse_coordinate_px'],done['baseline_metrics']['rmse_coordinate_px'],abs_tol=1e-6):
            raise ValueError('Cached inner baseline metric differs.')
        provenance.append(dict(fold=fold, config=cfg, complete=done, predictions=predictions))
    return centers, stable_hash(provenance)


def context_grid(boxes, sizes, height, width):
    """Feature-cell centers mapped from the local crop into normalized full-image coordinates."""
    b, size = boxes.float(), sizes.float()
    xs = b[:,0,None,None]+(torch.arange(width,device=b.device)[None,None,:]+.5)*(768/width)
    ys = b[:,1,None,None]+(torch.arange(height,device=b.device)[None,:,None]+.5)*(768/height)
    gx = (2*xs/size[:,0,None,None]-1).expand(-1,height,-1)
    gy = (2*ys/size[:,1,None,None]-1).expand(-1,-1,width)
    return torch.stack((gx,gy),dim=-1)


class ContextRefiner(ResNet34CV):
    def __init__(self,weights=None,context=True):
        super().__init__(weights);self.use_context=bool(context)
        if self.use_context:
            self.context3=nn.Conv2d(256,256,1)
            self.context4=nn.Conv2d(1024,512,1)
            for layer in (self.context3,self.context4):
                nn.init.zeros_(layer.weight);nn.init.zeros_(layer.bias)

    def encode(self,image):
        b=self.backbone;x=b.maxpool(b.relu(b.bn1(b.conv1(image))))
        c1=b.layer1(x);c2=b.layer2(c1);c3=b.layer3(c2);c4=b.layer4(c3)
        return c1,c2,c3,c4

    def aligned(self,feature,boxes,sizes,target):
        grid=context_grid(boxes,sizes,*target.shape[-2:])
        # Explicit fp32 sampling avoids mixed-dtype grid/input surprises under AMP.
        with torch.autocast(device_type=feature.device.type,enabled=False):
            value=F.grid_sample(feature.float(),grid,mode='bilinear',padding_mode='zeros',align_corners=False)
        return value.to(target.dtype)

    def forward(self,local,global_view,boxes,sizes):
        c1,c2,c3,c4=self.encode(local)
        if self.use_context:
            _,_,g3,g4=self.encode(global_view)
            c3=c3+self.context3(self.aligned(g3,boxes,sizes,c3))
            aligned=self.aligned(g4,boxes,sizes,c4)
            pooled=g4.mean(dim=(-2,-1),keepdim=True).expand(-1,-1,*c4.shape[-2:])
            c4=c4+self.context4(torch.cat((aligned,pooled),dim=1))
        return self.head(self.d1(self.d2(self.d3(c4,c3),c2),c1))[:,0]


def native_augmentation(box,flip,affine):
    # The local384 transformation and global256 transformation describe the same native frame.
    native_from_patch=np.array([[2.,0.,box[0]],[0.,2.,box[1]],[0.,0.,1.]])
    reflection=np.array([[-1.,0.,384.],[0.,1.,0.],[0.,0.,1.]]) if flip else np.eye(3)
    return native_from_patch@affine@reflection@np.linalg.inv(native_from_patch)


class ContextPatchDataset(Dataset):
    def __init__(self,records,cache,config,training,centers=None):
        self.records,self.cache,self.config=list(records),cache,config
        self.training,self.centers=training,centers
        ids={int(r['image_id']) for r in records}
        if len(ids)!=len(records):raise ValueError('Repeated patch ID.')
        if training:
            if ids!=set(config['train_ids']) or ids&set(config['forbidden_ids']):raise ValueError('Training IDs differ.')
        elif centers is None or ids!=set(centers):raise ValueError('Prediction centers/IDs differ.')
        if not hasattr(cache,'context_views'):cache.context_views={}
        self.tensor=transforms.Compose([transforms.ToTensor(),transforms.Normalize([.485,.456,.406],[.229,.224,.225])])
        self.jitter=transforms.ColorJitter(brightness=.15,contrast=.15,saturation=.08,hue=.01)

    def __len__(self):return len(self.records)

    def __getitem__(self,index):
        row=self.records[index];key=int(row['image_id']);image=self.cache.get(row)
        if key not in self.cache.context_views:
            self.cache.context_views[key]=image.resize((256,256),Image.Resampling.BILINEAR)
        global_view=self.cache.context_views[key].copy()
        if self.training:
            dx,dy=center_offset('small',[max(1e-10,min(1-1e-10,random.random())) for _ in range(3)])
            center=(float(row['x'])+dx,float(row['y'])+dy)
        else:
            c=self.centers[key];center=(float(c['Fovea_X']),float(c['Fovea_Y']))
        box=crop_geometry(center,768)
        local=image.crop(box).resize((384,384),Image.Resampling.BILINEAR)
        point=np.array(point_to_patch((row['x'],row['y']),box) if self.training else (.5,.5),dtype=np.float32)
        if self.training:
            flip=random.random()<.5
            if flip:local=local.transpose(Image.Transpose.FLIP_LEFT_RIGHT);point[0]=1-point[0]
            angle,scale=math.radians(random.uniform(-8,8)),random.uniform(.95,1.05)
            a,b=scale*math.cos(angle),-scale*math.sin(angle)
            tx,ty=random.uniform(-.02,.02)*384,random.uniform(-.02,.02)*384
            affine=np.array([[a,b,192+tx-a*192-b*192],[-b,a,192+ty+b*192-a*192],[0.,0.,1.]])
            point=((affine@[point[0]*384,point[1]*384,1.])[:2]/384).astype(np.float32)
            if not np.isfinite(point).all() or not ((point>0).all() and (point<1).all()):raise ValueError('Target left patch.')
            local=local.transform((384,384),Image.Transform.AFFINE,np.linalg.inv(affine)[:2].reshape(-1),resample=Image.Resampling.BILINEAR,fillcolor=(0,0,0))
            scaled=np.diag([256/int(row['width']),256/int(row['height']),1.])
            global_forward=scaled@native_augmentation(box,flip,affine)@np.linalg.inv(scaled)
            global_view=global_view.transform((256,256),Image.Transform.AFFINE,np.linalg.inv(global_forward)[:2].reshape(-1),resample=Image.Resampling.BILINEAR,fillcolor=(0,0,0))
            # torchvision ColorJitter draws use CPU Torch RNG; consume one draw sequence for both views.
            before=torch.get_rng_state();local=self.jitter(local);after=torch.get_rng_state()
            torch.set_rng_state(before);global_view=self.jitter(global_view);torch.set_rng_state(after)
        return self.tensor(local),self.tensor(global_view),torch.from_numpy(point),torch.tensor(box,dtype=torch.int64),torch.tensor([row['width'],row['height']],dtype=torch.float32)


def context_config(protocol,fold,arm,role,train_ids,evaluation_ids,forbidden_ids,epochs):
    cfg=dict(version=CONTEXT_VERSION,source_sha256=protocol['source_sha256'],coarse_fingerprint=protocol['coarse_fingerprint'],
        inner_fingerprint=protocol['inner_fingerprint'],outer_fold=fold,arm=arm,role=role,seed=(42000 if role=='inner' else 43000)+fold,
        train_ids=list(train_ids),evaluation_ids=list(evaluation_ids),forbidden_ids=list(forbidden_ids),epochs=int(epochs),
        schedule_epochs=protocol['inner_epochs'],image_size=384,global_size=256,crop_side=768,prior_sigma=48.,max_shift=256.)
    validate_context_config(cfg);return cfg


def validate_context_config(cfg):
    sets=[set(cfg[k]) for k in ('train_ids','evaluation_ids','forbidden_ids')]
    if any(len(s)!=len(cfg[k]) for s,k in zip(sets,('train_ids','evaluation_ids','forbidden_ids'))) or not sets[0] or any(sets[i]&sets[j] for i,j in ((0,1),(0,2),(1,2))):raise ValueError('Overlapping/repeated phase IDs.')
    if cfg['arm'] not in CONTEXT_ARMS or cfg['role'] not in ('inner','outer') or bool(sets[1])!=(cfg['role']=='inner'):raise ValueError('Invalid phase role.')
    if not 1<=cfg['epochs']<=cfg['schedule_epochs'] or (cfg['image_size'],cfg['global_size'],cfg['crop_side'])!=(384,256,768):raise ValueError('Invalid epoch/geometry.')
    if cfg['prior_sigma']!=48. or cfg['max_shift']!=256.:raise ValueError('This comparison fixes the decoder prior and fallback threshold.')


def load_context_checkpoint(path,signature):
    state=torch.load(path,map_location='cpu',weights_only=True)
    if state['signature']!=signature:raise ValueError('Checkpoint signature differs.')
    return state


def context_prediction_fold(cfg,centers,outer_fold):
    if cfg['role'] not in ('inner','outer') or int(cfg['outer_fold'])!=int(outer_fold):
        raise ValueError('Prediction role or outer fold differs from the phase.')
    expected=-1 if cfg['role']=='inner' else int(outer_fold)
    if not centers or any(int(row['fold'])!=expected for row in centers.values()):
        raise ValueError('Coarse center fold differs from the prediction phase.')
    return expected


@torch.inference_mode()
def predict_context(model,loader,cfg,centers,fold):
    prediction_fold=context_prediction_fold(cfg,centers,fold)
    model.eval();rows=[];offset=0
    for local,global_view,_,boxes,sizes in loader:
        with torch.autocast(device_type='cuda',enabled=True):
            logits=model(local.to('cuda:0'),global_view.to('cuda:0'),boxes.to('cuda:0'),sizes.to('cuda:0'))
        if logits.shape!=(len(local),96,96) or not torch.isfinite(logits).all():raise RuntimeError('Invalid context heatmap.')
        records=loader.dataset.records[offset:offset+len(local)]
        points=[[float(centers[int(r['image_id'])]['Fovea_X']),float(centers[int(r['image_id'])]['Fovea_Y'])] for r in records]
        decoded=decode_heatmap(coarse_log_prior(logits,boxes,points,48.),radius=3).cpu().tolist()
        for record,point,box in zip(records,decoded,boxes.tolist()):
            row=make_refinement_row(record,point,centers[int(record['image_id'])],box,prediction_fold,256.)
            row.update(decoder='prior',refine_applied=1,arm=cfg['arm']);rows.append(row)
        offset+=len(local)
    if offset!=len(loader.dataset):raise ValueError('Inference coverage differs.')
    return rows


def gpu_context_preflight(weights,output):
    seed_everything(44000);local=ContextRefiner(weights,False).to('cuda:0').eval()
    seed_everything(44000);context=ContextRefiner(weights,True).to('cuda:0').eval()
    x=torch.randn(1,3,384,384,device='cuda:0');g=torch.randn(1,3,256,256,device='cuda:0')
    box=torch.tensor([[1100,600,1868,1368]],device='cuda:0');size=torch.tensor([[2992,2000]],device='cuda:0')
    with torch.no_grad(),torch.autocast(device_type='cuda'):
        expected=local(x,g,box,size);actual=context(x,g,box,size)
    difference=float((actual-expected).abs().max())
    if difference>1e-3 or actual.shape!=(1,96,96):raise RuntimeError('Zero-initialized context changed the local starting function.')
    del local,expected,actual;context.train();context.zero_grad(set_to_none=True)
    with torch.autocast(device_type='cuda'):logits=context(x,g,box,size)
    loss=heatmap_loss(logits,torch.tensor([[.5,.5]],device='cuda:0'),1.5);loss.backward()
    gradients={name:bool(p.grad is not None and torch.isfinite(p.grad).all() and p.grad.abs().sum()>0)
        for name,p in context.named_parameters() if name in ('context3.weight','context4.weight')}
    if not math.isfinite(float(loss.detach())) or not all(gradients.values()):raise RuntimeError('Global fusion gradient check failed.')
    if any(m.training for m in context.backbone.modules() if isinstance(m,nn.BatchNorm2d)):raise RuntimeError('Backbone BatchNorm not frozen.')
    save_json(output/'gpu_preflight.json',dict(passed=True,zero_init_max_abs_difference=difference,gradients=gradients,loss=float(loss.detach())))
    del context,x,g,box,size,logits,loss;gc.collect();torch.cuda.empty_cache()


def train_context_phase(records,cache,cfg,folder,weights,evaluator=None):
    validate_context_config(cfg)
    if (evaluator is not None)!=(cfg['role']=='inner'):raise ValueError('Outer training must not have an evaluation callback.')
    folder.mkdir(parents=True,exist_ok=True);signature=stable_hash(cfg)
    if (folder/'config.json').exists() and json.loads((folder/'config.json').read_text())!=cfg:raise ValueError('Phase config differs.')
    save_json(folder/'config.json',cfg)
    if (folder/'phase_complete.json').exists():
        done=json.loads((folder/'phase_complete.json').read_text())
        checkpoint=folder/('best.pt' if evaluator else 'final.pt')
        if done['signature']!=signature or not checkpoint.is_file():raise ValueError('Restore the full v4 Output, not a light ZIP.')
        return done
    seed_everything(cfg['seed']);torch.set_num_threads(2)
    dataset=ContextPatchDataset(records,cache,cfg,True);loader=DataLoader(dataset,batch_size=8,shuffle=True,num_workers=0)
    last=folder/'last.pt';model=ContextRefiner(None if last.exists() else weights,cfg['arm']=='context').to('cuda:0')
    backbone_ids={id(p) for p in model.backbone.parameters()}
    optimizer=torch.optim.AdamW([dict(params=model.backbone.parameters(),lr=5e-5),
        dict(params=[p for p in model.parameters() if id(p) not in backbone_ids],lr=5e-4)],weight_decay=1e-4)
    scheduler=torch.optim.lr_scheduler.LambdaLR(optimizer,lambda step:.05+.95*(1+math.cos(math.pi*step/cfg['schedule_epochs']))/2)
    scaler=torch.amp.GradScaler('cuda');start=0;history=[];best=dict(score=float('inf'),epoch=0);best_weights=None
    if last.exists():
        state=torch.load(last,map_location='cpu',weights_only=False)
        if state['signature']!=signature:raise ValueError('Optimizer signature differs.')
        model.load_state_dict(state['model'],strict=True);optimizer.load_state_dict(state['optimizer']);scheduler.load_state_dict(state['scheduler']);scaler.load_state_dict(state['scaler'])
        start,history,best=state['epoch'],state['history'],state['best'];best_weights=state['best_model'];del state
        # Restore the best model belonging to this committed optimizer/history prefix.
        # A crash after a later best.pt write must not mix an uncommitted future best with this prefix.
        if evaluator and best['epoch']:
            atomic_torch_save(folder/'best.pt',dict(model=best_weights,signature=signature,epoch=best['epoch'],selection=best,config=cfg))
    for epoch in range(start+1,cfg['epochs']+1):
        # Additional global layers must not change the paired minibatches or augmentation draws.
        seed_everything(cfg['seed']+1000*epoch);begin=time.perf_counter();model.train();total=count=skipped=0
        for local,global_view,points,boxes,sizes in loader:
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda'):
                logits=model(local.to('cuda:0'),global_view.to('cuda:0'),boxes.to('cuda:0'),sizes.to('cuda:0'))
                if logits.shape!=(len(local),96,96):raise RuntimeError('Fine shape differs.')
                loss=heatmap_loss(logits,points.to('cuda:0'),1.5)
            if not torch.isfinite(loss):raise RuntimeError('Non-finite loss.')
            scaler.scale(loss).backward();scaler.unscale_(optimizer);torch.nn.utils.clip_grad_norm_(model.parameters(),5.)
            old_scale=scaler.get_scale();scaler.step(optimizer);scaler.update();skipped+=int(scaler.get_scale()<old_scale)
            total+=float(loss.detach())*len(local);count+=len(local)
        if skipped==len(loader) and history and history[-1]['amp_skips']==len(loader):raise RuntimeError('All optimizer updates skipped in consecutive epochs.')
        row=dict(epoch=epoch,train_ce=total/count,amp_skips=skipped,lr_backbone=optimizer.param_groups[0]['lr'])
        if evaluator:
            values=evaluator(model);row.update(values)
            if values['score']<best['score']:
                best=dict(values,epoch=epoch)
                best_weights={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
                atomic_torch_save(folder/'best.pt',dict(model=best_weights,signature=signature,epoch=epoch,selection=best,config=cfg))
        scheduler.step();row['seconds']=time.perf_counter()-begin;history.append(row)
        if epoch%5==0 or epoch==cfg['epochs']:
            atomic_torch_save(last,dict(model=model.state_dict(),optimizer=optimizer.state_dict(),scheduler=scheduler.state_dict(),scaler=scaler.state_dict(),signature=signature,epoch=epoch,history=history,best=best,best_model=best_weights))
        atomic_csv(folder/'history.csv',history);save_json(folder/'status.json',dict(state='training',epoch=epoch,signature=signature))
        print(f'Outer{cfg["outer_fold"]} {cfg["role"]}/{cfg["arm"]} {epoch:02}/{cfg["epochs"]} CE={total/count:.4f} {row["seconds"]:.1f}s',flush=True)
    if not evaluator:atomic_torch_save(folder/'final.pt',dict(model=model.state_dict(),signature=signature,epoch=cfg['epochs'],config=cfg))
    done=dict(signature=signature,best=best if evaluator else {},epochs=cfg['epochs']);save_json(folder/'phase_complete.json',done)
    if last.exists():last.unlink()
    del model,loader,dataset,best_weights;gc.collect();torch.cuda.empty_cache();return done


def context_design(prepared,fold):
    dev,fit,held,outer=inner_partition(prepared['train'],fold)
    design=dict(protocol=prepared['protocol'],fold=fold,fit_ids=list(map(int,fit.image_id)),inner_val_ids=list(map(int,held.image_id)),outer_ids=list(map(int,outer.image_id)))
    return dev,fit,held,outer,design


def run_context_fold(prepared,fold,weights,cache):
    dev,fit,held,outer,design=context_design(prepared,fold)
    job=prepared['output']/f'outer_{fold}';job.mkdir(parents=True,exist_ok=True);signature=stable_hash(design)
    if (job/'fold_complete.json').exists():
        done=json.loads((job/'fold_complete.json').read_text())
        if done['signature']!=signature or any(not (job/p).is_file() for p in done['required_models']):raise ValueError('Completed fold lacks full weights or differs.')
        return
    save_json(job/'design.json',design)
    atomic_csv(job/'split_roles.csv',[dict(image_id=int(r['image_id']),pixel_sha256=r['pixel_sha256'],role=role) for role,frame in [('inner_train',fit),('inner_validation',held),('outer_validation',outer)] for r in frame.to_dict('records')])
    centers=prepared['inner'][fold];baseline=metrics_from_rows(list(centers.values()))['rmse_coordinate_px'];decisions={}
    for arm in CONTEXT_ARMS:
        cfg=context_config(prepared['protocol'],fold,arm,'inner',design['fit_ids'],design['inner_val_ids'],design['outer_ids'],prepared['protocol']['inner_epochs'])
        validation=DataLoader(ContextPatchDataset(held.to_dict('records'),cache,cfg,False,centers),batch_size=8,shuffle=False,num_workers=0)
        def evaluate(model):return inner_mode_metrics(predict_context(model,validation,cfg,centers,fold),centers)
        done=train_context_phase(fit.to_dict('records'),cache,cfg,job/f'inner_{arm}',weights,evaluate)
        decision=selected_inner_epoch(done['best'],baseline,.05);decisions[arm]=decision;del validation
        if decision['enabled']:
            cfg=context_config(prepared['protocol'],fold,arm,'outer',list(map(int,dev.image_id)),[],design['outer_ids'],decision['epoch'])
            folder=job/f'outer_{arm}';train_context_phase(dev.to_dict('records'),cache,cfg,folder,weights)
            model=ContextRefiner(None,arm=='context').to('cuda:0');state=load_context_checkpoint(folder/'final.pt',stable_hash(cfg))
            model.load_state_dict(state['model'],strict=True);del state
        for subset,frame,prior in [('validation',outer,prepared['coarse']['validation'][fold]),('test',prepared['test'],prepared['coarse']['test'][fold])]:
            records=frame.to_dict('records')
            if decision['enabled']:
                loader=DataLoader(ContextPatchDataset(records,cache,cfg,False,prior),batch_size=8,shuffle=False,num_workers=0)
                predictions=predict_context(model,loader,cfg,prior,fold);del loader
                for row in predictions:row['selection_reason']=decision['reason']
            else:predictions=identity_rows(records,prior,fold,decision)
            for row in predictions:row['arm']=arm
            atomic_csv(job/f'{subset}_{arm}.csv',predictions)
        if decision['enabled']:del model
        gc.collect();torch.cuda.empty_cache()
    save_json(job/'inner_decisions.json',decisions)
    required=sorted(p.relative_to(job).as_posix() for p in job.rglob('*.pt'))
    save_json(job/'fold_complete.json',dict(signature=signature,decisions=decisions,required_models=required,predictions_complete=True,outer_labels_used_for_epoch_selection=False))


def context_reports(prepared):
    output=prepared['output'];available=[f for f in range(5) if (output/f'outer_{f}/fold_complete.json').exists()]
    selection=dict(complete=len(available)==5 and not prepared['protocol']['smoke_test'],completed_folds=available,smoke_test=prepared['protocol']['smoke_test'],
        official_metric_confirmed=True,metric='rmse_coordinate_px',selected_by_outer_scores=False,candidate_rule='Predeclared context_equal; inner 5% gain gate',
        candidate_public_score=None,agent_submitted_to_competition=False,limitation='Legacy fixed coarse checkpoints used outer labels for stopping. This is conditional development evaluation, not fully independent evaluation of the complete pipeline.')
    if not selection['complete']:save_json(output/'selection.json',selection);return False
    records,testing=prepared['train'].to_dict('records'),prepared['test'].to_dict('records');coarse=prepared['coarse']
    baseline=validate_oof([r for f in range(5) for r in coarse['validation'][f].values()],records)
    atomic_csv(output/'oof_coarse512.csv',baseline)
    write_submission(prepared['data']/'sample_submission.csv',average_test_folds({f:list(coarse['test'][f].values()) for f in range(5)},testing),output/'submission_coarse512.csv')
    comparisons=[dict(arm='coarse512',**metrics_from_rows(baseline))];decision_rows=[]
    for arm in CONTEXT_ARMS:
        predictions=[];test_folds={};active=0
        for f in range(5):
            job=output/f'outer_{f}';done=json.loads((job/'fold_complete.json').read_text());design=context_design(prepared,f)[-1]
            if done['signature']!=stable_hash(design) or not done['predictions_complete'] or done['outer_labels_used_for_epoch_selection']:raise ValueError('Fold provenance differs.')
            decision=done['decisions'][arm];expected=selected_inner_epoch(dict(score=decision['inner_score'],epoch=decision['inner_best_epoch']),decision['inner_baseline'],.05)
            if decision!=expected:raise ValueError('Inner selection differs.')
            active+=int(decision['enabled']);decision_rows.append(dict(arm=arm,outer_fold=f,**decision))
            held=prepared['train'][prepared['train'].fold==f].to_dict('records');val=validate_oof(read_rows(job/f'validation_{arm}.csv'),held)
            validate_test_rows(val,held,f);predictions.extend(val);test_folds[f]=read_rows(job/f'test_{arm}.csv');validate_test_rows(test_folds[f],testing,f)
        predictions=validate_oof(predictions,records)
        for suffix,preds,folds in [('refined',predictions,test_folds),('equal',blend_prediction_rows(baseline,predictions,True),{f:blend_prediction_rows(list(coarse['test'][f].values()),test_folds[f],False) for f in range(5)})]:
            name=f'{arm}_{suffix}';atomic_csv(output/f'oof_{name}.csv',preds);mean=average_test_folds(folds,testing)
            atomic_csv(output/f'test_{name}.csv',mean);write_submission(prepared['data']/'sample_submission.csv',mean,output/f'submission_{name}.csv')
            comparisons.append(dict(arm=name,active_folds=active,**metrics_from_rows(preds)))
        raw=raw_prediction_rows(predictions);atomic_csv(output/f'oof_{arm}_raw.csv',raw);comparisons.append(dict(arm=f'{arm}_raw_diagnostic',active_folds=active,**metrics_from_rows(raw)))
    atomic_csv(output/'comparison.csv',comparisons,fields=['arm','active_folds','n','mean_distance_px','median_distance_px','p90_distance_px','max_distance_px','mae_x_px','mae_y_px','rmse_coordinate_px'])
    atomic_csv(output/'inner_decisions.csv',decision_rows)
    shutil.copy2(output/'submission_context_equal.csv',output/'submission_context_v4_candidate.csv')
    selection.update(candidate='submission_context_v4_candidate.csv',public_best_reference=12.31624,public_coarse_reference=14.16538)
    save_json(output/'selection.json',selection);print(pd.DataFrame(comparisons).to_string(index=False),flush=True);return True


def context_archive(output,full=False):
    target=output.parent/'task3_context_v4_full_backup.zip' if full else output/'task3_context_v4_results_light.zip'
    temporary=target.with_suffix('.zip.tmp')
    with zipfile.ZipFile(temporary,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=1,allowZip64=True) as archive:
        for p in sorted(output.rglob('*')):
            if p.is_file() and not p.is_symlink() and p.suffix not in ('.tmp','.zip') and (full or p.suffix in ('.csv','.json','.md')):
                archive.write(p,p.relative_to(output))
    temporary.replace(target)
    if full:
        with zipfile.ZipFile(target) as archive:
            names=set(archive.namelist())
            for f in range(5):
                done=json.loads((output/f'outer_{f}/fold_complete.json').read_text())
                if any(f'outer_{f}/{name}' not in names for name in done['required_models']):raise RuntimeError('Full backup lacks required weights.')
    return target


def prepare_context(source,data_dir=None,output_dir='/kaggle/working/task3_context_refine_v4',resume_dir=None,smoke=False):
    if not Path('/kaggle/working').is_dir() or not torch.cuda.is_available():raise RuntimeError('Kaggle GPU required. No local training was started.')
    data,output=find_data(data_dir),Path(output_dir).resolve();source=Path(source)
    manifest=load_manifest(data);train=assign_folds(manifest[manifest.subset=='train'],42,5);test=manifest[manifest.subset=='test'].copy()
    if len(train)!=80 or len(test)!=20:raise ValueError('Expected80/20 images.')
    coarse=validate_coarse_bundle(source/'coarse',train.to_dict('records'),test.to_dict('records'))
    centers,inner_fingerprint=validate_context_centers(source,train,coarse['fingerprint'])
    protocol=dict(version=CONTEXT_VERSION,source_sha256=dict(CONTEXT_SOURCE_SHA256),coarse_fingerprint=coarse['fingerprint'],inner_fingerprint=inner_fingerprint,
        inner_epochs=2 if smoke else 60,arms=list(CONTEXT_ARMS),prior_sigma=48.,max_shift=256.,min_inner_gain=.05,candidate_variant='context_equal',smoke_test=bool(smoke),official_metric='rmse_coordinate_px',
        augmentation='v3 small jitter + paired native affine/global view + shared ColorJitter; epoch-keyed RNG',global_size=256,context_initialization='zero additive projections',legacy_coarse_outer_selection_bias_remains=True)
    output.mkdir(parents=True,exist_ok=True)
    if shutil.disk_usage(output).free<9*1024**3:raise RuntimeError('Need9 GiB free for weights, atomic saves and full backup.')
    restore_backup(resume_dir,output,protocol)
    if (output/'protocol.json').exists() and json.loads((output/'protocol.json').read_text())!=protocol:raise ValueError('Output protocol differs; use a new directory.')
    save_json(output/'protocol.json',protocol);train.to_csv(output/'split.csv',index=False);manifest.to_csv(output/'manifest.csv',index=False)
    save_json(output/'environment.json',dict(torch=torch.__version__,numpy=np.__version__,gpu=torch.cuda.get_device_name(0),visible_gpu_count=torch.cuda.device_count(),device='cuda:0 sequential',grid_sample_backward='CUDA backward may be nondeterministic; bitwise reruns are not promised'))
    for f in range(5):
        folder=output/f'inner_cache/fold_{f}';folder.mkdir(parents=True,exist_ok=True)
        atomic_csv(folder/'inner_validation.csv',list(centers[f].values()))
    save_json(output/'cache_provenance.json',dict(coarse_fingerprint=coarse['fingerprint'],inner_fingerprint=inner_fingerprint,input_payload_sha256=CONTEXT_CACHE_SHA256))
    return dict(data=data,output=output,train=train,test=test,coarse=coarse,inner=centers,protocol=protocol)


def run_context(prepared,run_folds=(0,1,2,3,4),weights=None,full_backup=True):
    if not Path('/kaggle/working').is_dir() or not torch.cuda.is_available():raise RuntimeError('Kaggle GPU required.')
    if not run_folds or len(set(run_folds))!=len(run_folds) or not set(run_folds)<=set(range(5)):raise ValueError('Invalid fold schedule.')
    output=prepared['output'];cache=OriginalImageCache(prepared['data'])
    pending=any(not (output/f'outer_{f}/fold_complete.json').exists() for f in run_folds)
    weights=resolve_weights(output,weights) if pending else None
    with training_lock(output):
        try:
            if pending:gpu_context_preflight(weights,output)
            for f in run_folds:
                save_json(output/'status.json',dict(state='running',outer_fold=f,updated_unix=time.time()))
                run_context_fold(prepared,f,weights,cache);context_reports(prepared);context_archive(output)
            complete=context_reports(prepared)
            save_json(output/'status.json',dict(state='completed' if complete else 'partial_completed',updated_unix=time.time()));context_archive(output)
            if complete and full_backup:print('Full backup:',context_archive(output,True),flush=True)
        except BaseException as error:
            save_json(output/'status.json',dict(state='interrupted' if isinstance(error,KeyboardInterrupt) else 'failed',error=str(error),updated_unix=time.time()))
            context_archive(output);raise
    return output


In [ ]:
CONTEXT_CACHE_B64='...'
INPUT_CACHE=unpack_context_inputs(CONTEXT_CACHE_B64,CONTEXT_CACHE_SHA256,Path('/kaggle/working/task3_context_v4_input_cache'))
PREPARED=prepare_context(INPUT_CACHE,data_dir=DATA_DIR,output_dir=OUTPUT_DIR,resume_dir=RESUME_DIR,smoke=SMOKE_TEST)
print('GPU:',torch.cuda.get_device_name(0))
print('固定粗定位:',PREPARED['coarse']['baseline_metrics'])
for f in RUN_FOLDS:
    dev,fit,held,outer=inner_partition(PREPARED['train'],f)
    baseline=metrics_from_rows(list(PREPARED['inner'][f].values()))['rmse_coordinate_px']
    print(f'Outer{f}: inner fit={len(fit)}, inner validation={len(held)}, outer refit={len(dev)}, outer evaluation={len(outer)}, inner coarse RMSE={baseline:.4f}')


## 2. 训练与评价

局部768原图裁剪缩到384，整图缩到256。v3的小扰动、随机翻转、±8度旋转、缩放、平移与颜色增强保留；整图使用同一个原图几何变换，颜色增强共享随机参数。两组逐轮重设相同数据随机数，避免额外网络层改变批次或增强。全局c3/c4特征通过grid_sample与局部区域对齐，附加全局池化特征，融合投影从零初始化。骨干BatchNorm冻结。

两组均使用σ48原图像素的位置先验和256像素位移回退，后者为异常回退阈值，前者是软约束。先验不保证总误差小于48像素。内层按固定粗精50/50系统的官方坐标RMSE选轮次，只有相对内层粗基线至少改善5%才启用；否则该折保持粗预测。

每折内层约51～52训练/12～13验证；对应粗中心来自只见过内层训练图的v3辅助模型。保留折2内层0070严重失败的真实预测，不替换、不删样本。外层64图从ImageNet重训所选轮数，保持60轮学习率周期，不使用外层16图选轮次。

与v3相比，本版按轮次固定数据随机数且整图视图同步增强；应以本版local组判断新增上下文的作用，不把跨版本差值全部归因于融合。context和local各自用相同内层规则选轮次，因此这是两套完整训练协议的配对比较。

旧粗模型曾使用外层验证挑轮次，所以整条级联仍不是完全独立验证。本版外层不选择融合比例、轮次或候选。CUDA grid_sample反向可能非确定，固定种子不承诺逐比特重复。

开始训练前自动执行GPU前向/反向检查，验证零初始化等价、融合层梯度和冻结BN。SMOKE_TEST只做2轮短流程，独立目录且禁止正式候选。


In [ ]:
RESULT_DIR=run_context(PREPARED,run_folds=RUN_FOLDS,weights=WEIGHTS,full_backup=CREATE_FULL_BACKUP)


## 3. 导出与保存

全部五折完成且非smoke时，预先指定context_equal生成submission_context_v4_candidate.csv，是否启用精修由各折内层决定。候选不会因外层结果排名改变，也不会自动提交。先交回轻量ZIP分析，再决定是否提交。

comparison.csv同时保存粗模型、每组原始精修、回退后精修、固定50/50系统的指标。raw诊断的关闭折仍是粗预测，不代表该折训练过精修器。

完整备份task3_context_v4_full_backup.zip默认生成，按每折required_models核对；建议下载到D盘。轻量ZIP不能恢复模型。恢复需挂载本版完整Output并设置RESUME_DIR，保持代码、数据和配置相同；每5轮保存optimizer断点与该断点对应的最佳权重，完成阶段保留best.pt或final.pt。最大权重数随内层门控变化。初始需至少9GiB空闲，原图缓存约2GiB内存，另有较小的整图缓存。


In [ ]:
from IPython.display import display
print((RESULT_DIR/'selection.json').read_text(encoding='utf-8'))
if (RESULT_DIR/'comparison.csv').exists():
    display(pd.read_csv(RESULT_DIR/'comparison.csv'))
    display(pd.read_csv(RESULT_DIR/'inner_decisions.csv'))
candidate=RESULT_DIR/'submission_context_v4_candidate.csv'
if candidate.exists():
    display(pd.read_csv(candidate))
    print('预设候选:',candidate)
print('交回分析:',RESULT_DIR/'task3_context_v4_results_light.zip')
print('完整备份:',RESULT_DIR.parent/'task3_context_v4_full_backup.zip')
